# GridapTensorProduct.jl — Library Walk-Through

This notebook traces the **five-stage pipeline** of GridapTensorProduct.jl from raw 1D
meshes all the way to a solved linear system, explaining how each structure is built and
how the stages connect.

**Domain:** 2D Poisson on Ω = [0,1]² with homogeneous Dirichlet BCs and unit forcing:
$$-\Delta u = 1 \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

---
## Architecture overview

```
Stage 1  TensorProductMeasure        — bundle N subdomain quadrature measures
Stage 2  TensorProductFESpace        — bundle N subdomain FE spaces + DOF mapping
Stage 3  TensorProductGeometry       — triangulation and cell-field wrappers
Stage 4  TensorProductFEMOperators   — extract 1D matrices; form Kronecker global ops
Stage 5  TensorProductAssembly       — high-level API; assemble and return (A, b)
```

Key idea: **work with 1D subdomains independently in Stages 1–4; couple via Kronecker products only at assembly (Stage 5).**

In [1]:
using Gridap
using GridapTensorProduct
import GridapTensorProduct: ⊗
using LinearAlgebra

---
## Stage 1 — 1D meshes and subdomain measures

Everything starts with ordinary Gridap objects on **1D subdomains**. No tensor-product types yet.

In [2]:
Nx, Ny = 4, 3
model_x = CartesianDiscreteModel((0.0, 1.0), Nx)
model_y = CartesianDiscreteModel((0.0, 1.0), Ny)
Ωx = Interior(model_x)
Ωy = Interior(model_y)

quad_order = 2
dΩx = Measure(Ωx, quad_order)
dΩy = Measure(Ωy, quad_order)

println("Subdomain x: ", num_cells(Ωx), " cells")
println("Subdomain y: ", num_cells(Ωy), " cells")

Subdomain x: 4 cells
Subdomain y: 3 cells


### TensorProductMeasure — the ⊗ operator

Writing `dΩx ⊗ dΩy` creates a `TensorProductMeasure` that bundles the two measures.
When a user writes `∫(expr) * (dΩx ⊗ dΩy)`, the library intercepts it and creates
a `TensorProductIntegrand` marker — no actual integration happens at this point.
Assembly is deferred to Stage 5.

In [3]:
dΩ_tp = dΩx ⊗ dΩy

println(typeof(dΩ_tp))           # TensorProductMeasure
println(num_subdomains(dΩ_tp))   # 2

# Intercepted multiplication creates a deferred marker
marker = ∫(x -> 1.0) * dΩ_tp
println(typeof(marker))           # TensorProductIntegrand

TensorProductMeasure{2, Tuple{Gridap.CellData.GenericMeasure, Gridap.CellData.GenericMeasure}}
2
TensorProductIntegrand


---
## Stage 2 — Subdomain FE spaces and the tensor DOF map

Each subdomain gets its own standard Gridap `TestFESpace` / `TrialFESpace`.
Dirichlet BCs are applied **per subdomain** before any tensor coupling.

**DOF ordering convention** (subdomain 1 = fastest-varying):
$$\text{global\_dof}(i_1, i_2) = i_1 + (i_2 - 1) \cdot n_{f,1}$$

This matches the Kronecker convention `kron(M_2, M_1)` where M_1 is innermost.

In [4]:
reffex = ReferenceFE(lagrangian, Float64, 1)
reffey = ReferenceFE(lagrangian, Float64, 2)

Vx = TestFESpace(Ωx, reffex; conformity=:H1, dirichlet_tags="boundary")
Vy = TestFESpace(Ωy, reffey; conformity=:H1, dirichlet_tags="boundary")
Ux = TrialFESpace(Vx, 0.0)
Uy = TrialFESpace(Vy, 0.0)

println("Free DOFs on Ωx: ", num_free_dofs(Vx))   
println("Free DOFs on Ωy: ", num_free_dofs(Vy))   

Free DOFs on Ωx: 3
Free DOFs on Ωy: 5


In [5]:
V_tp = TensorProductFESpace(Vx, Vy)
U_tp = TensorProductFESpace(Ux, Uy)

n_tp = num_free_dofs(V_tp)   # = num_free_dofs(Vx) * num_free_dofs(Vy)
println("Free DOFs: ", num_free_dofs(Vx), " × ", num_free_dofs(Vy), " = ", n_tp)

# Satisfies Gridap's FESpace interface
println(get_free_dof_ids(V_tp))   # Base.OneTo(49)

Free DOFs: 3 × 5 = 15
Base.OneTo(15)


In [6]:
get_cell_ids(V_tp)   # Vector of cell IDs build as tuples of cell ids from each subdomain

12-element Gridap.Arrays.Table{Int32, Vector{Int32}, Vector{Int32}}:
 [1, 1]
 [2, 1]
 [3, 1]
 [4, 1]
 [1, 2]
 [2, 2]
 [3, 2]
 [4, 2]
 [1, 3]
 [2, 3]
 [3, 3]
 [4, 3]

In [7]:
get_cell_dof_ids(V_tp)   # Vector of vectors of DOF IDs for each cell, with DOF IDs as tuples of DOF ids from each subdomain

12-element Gridap.Arrays.Table{Vector{Int32}, Vector{Vector{Int32}}, Vector{Int32}}:
 [[-1, -1], [1, -1], [-1, 1], [1, 1], [-1, 3], [1, 3]]
 [[1, -1], [2, -1], [1, 1], [2, 1], [1, 3], [2, 3]]
 [[2, -1], [3, -1], [2, 1], [3, 1], [2, 3], [3, 3]]
 [[3, -1], [-2, -1], [3, 1], [-2, 1], [3, 3], [-2, 3]]
 [[-1, 1], [1, 1], [-1, 2], [1, 2], [-1, 4], [1, 4]]
 [[1, 1], [2, 1], [1, 2], [2, 2], [1, 4], [2, 4]]
 [[2, 1], [3, 1], [2, 2], [3, 2], [2, 4], [3, 4]]
 [[3, 1], [-2, 1], [3, 2], [-2, 2], [3, 4], [-2, 4]]
 [[-1, 2], [1, 2], [-1, -2], [1, -2], [-1, 5], [1, 5]]
 [[1, 2], [2, 2], [1, -2], [2, -2], [1, 5], [2, 5]]
 [[2, 2], [3, 2], [2, -2], [3, -2], [2, 5], [3, 5]]
 [[3, 2], [-2, 2], [3, -2], [-2, -2], [3, 5], [-2, 5]]

---
## Stage 3 — Tensor-product geometry

`TensorProductTriangulation` wraps N component triangulations.
Its `num_cells` equals the product of component cell counts.
Access individual components with integer indexing.

In [8]:
trian_tp = get_triangulation(V_tp)

println(typeof(trian_tp))                             # TensorProductTriangulation
println("Total tensor cells: ", num_cells(trian_tp))  # 8 * 8 = 64
println("Cells on Ωx: ",        num_cells(trian_tp[1]))
println("Cells on Ωy: ",        num_cells(trian_tp[2]))

# Plural accessor for direct tuple access
trians = get_triangulations(V_tp)
println("Number of components: ", length(trians))

TensorProductTriangulation{Tuple{Gridap.Geometry.BodyFittedTriangulation{1, 1, CartesianDiscreteModel{1, Float64, typeof(identity)}, CartesianGrid{1, Float64, typeof(identity)}, Gridap.Arrays.IdentityVector{Int64}}, Gridap.Geometry.BodyFittedTriangulation{1, 1, CartesianDiscreteModel{1, Float64, typeof(identity)}, CartesianGrid{1, Float64, typeof(identity)}, Gridap.Arrays.IdentityVector{Int64}}}}
Total tensor cells: 12
Cells on Ωx: 4
Cells on Ωy: 3
Number of components: 2


---
## Stage 4 — Subdomain operator extraction and Kronecker assembly

### 4a: Assemble fundamental 1D matrices

For each subdomain $k$, `assemble_subdomain_operators` calls Gridap's
`AffineFEOperator` to extract five matrices:

| Matrix | Weak form |
|--------|----------|
| $M^{(k)}$ | $\int_{\Omega_k} \varphi_i \varphi_j \, dx_k$ |
| $K^{(k)}$ | $\int_{\Omega_k} \nabla\varphi_i \cdot \nabla\varphi_j \, dx_k$ |
| $G^{(k)}$ | $\int_{\Omega_k} (\partial_x \varphi_i) \, \varphi_j \, dx_k$ |
| $D^{(k)}$ | $\int_{\Omega_k} \varphi_i \, (\partial_x \varphi_j) \, dx_k$ |
| $A^{(k)}$ | $\int_{\Omega_k} \varphi_i \, (b_k \partial_x \varphi_j) \, dx_k$ |

In [9]:
ops = assemble_subdomain_operators((Vx, Vy), (dΩx, dΩy))

println(typeof(ops))   # TensorProductSubdomainOperators{2}
println("M^(x) size: ", size(ops.M_ops[1]))  
println("K^(x) size: ", size(ops.K_ops[1]))  
println("M^(y) size: ", size(ops.M_ops[2]))  
println("K^(y) size: ", size(ops.K_ops[2])) 

TensorProductSubdomainOperators{2}
M^(x) size: (3, 3)
K^(x) size: (3, 3)
M^(y) size: (5, 5)
K^(y) size: (5, 5)


### 4b: Kronecker assembly

Global operators follow the **subdomain 1 = innermost** convention:

| Operator | N=2 formula |
|----------|-------------|
| Mass | `kron(My, Mx)` |
| Stiffness | `kron(My, Kx) + kron(Ky, Mx)` |
| Gradient | `[kron(My, Gx); kron(Gy, Mx)]` (block column) |

In [10]:
M_global = assemble_mass_tensor(ops)
A_global = assemble_stiffness_tensor(ops)
G_global = assemble_gradient_tensor(ops)

println("Global mass size:      ", size(M_global))  # (49, 49)
println("Global stiffness size: ", size(A_global))  # (49, 49)
println("Global gradient size:  ", size(G_global))  # (98, 49) — 2 blocks stacked
println("Stiffness symmetric:   ", maximum(abs, A_global - A_global') < 1e-14)

# Verify against direct kron
Mx, Kx = ops.M_ops[1], ops.K_ops[1]
My, Ky = ops.M_ops[2], ops.K_ops[2]
A_ref = kron(My, Kx) + kron(Ky, Mx)
println("Max error vs kron reference: ", maximum(abs, A_global - A_ref))

Global mass size:      (15, 15)
Global stiffness size: (15, 15)
Global gradient size:  (30, 15)
Stiffness symmetric:   true
Max error vs kron reference: 0.0


### Low-level API: `TensorProductOperator` (lazy + cached)

In [11]:
op_stiff = TensorProductOperator(:stiffness, (Vx, Vy), (dΩx, dΩy))

println("Cached before first call: ", op_stiff.global_matrix !== nothing)  # false
A_lazy = get_global_matrix(op_stiff)
println("Cached after first call:  ", op_stiff.global_matrix !== nothing)  # true
println("Same object on repeat:    ", A_lazy === get_global_matrix(op_stiff))

Cached before first call: false
Cached after first call:  true
Same object on repeat:    true


---
## Stage 5 — High-level assembly: `TensorProductAffineOperator`

The recommended entry point for most users. Lazily assembles `(A, b)` on first
access to `get_matrix` / `get_vector` and caches the result.

### Two APIs

**New API (recommended)** — bilinear form takes `(u, v, dΩ)` as arguments.
The library evaluates it on each 1D subdomain and *automatically determines* the
Kronecker decomposition (product rule vs. sum rule) without `op_type`.

**Legacy API** — `op_type` keyword selects the operator type explicitly (backward compatible).

### How auto-detection works

For each `(u,v,dΩ) -> ...` term:
1. Evaluate the form on each subdomain k → per-subdomain matrix `A_k`
2. Compare with the reference mass `M_k`:
   - All `A_k ≈ M_k` → **product rule**: `M₁⊗M₂⊗…` (mass)
   - Otherwise → **sum rule**: `Σ_k M_N⊗…⊗A_k⊗…⊗M₁` (stiffness, advection, …)

Only the matrices required by the form are assembled — no pre-computation of unused operators.

In [23]:
# ── New API: 3-argument form, no op_type needed ──────────────────────────────
# The library detects the Kronecker decomposition automatically.

a_new(u, v) = ∫(∇(u) ⋅ ∇(v)) * dΩ_tp   # bilinear form takes dΩ as 3rd argument
l_new(v)    = ∫(1.0 * v) * dΩ_tp       # linear form likewise

op_new = TensorProductAffineOperator(a_new, l_new, U_tp, V_tp)

println("weak_form set: ", op_new.weak_form !== nothing)   # true → uses auto-detection
println("op_type:       ", op_new.op_type)                  # nothing (not needed)
println("Cached before: ", op_new._matrix !== nothing)

weak_form set: true
op_type:       nothing
Cached before: false


In [21]:
A_new = get_matrix(op_new)
b_new = get_vector(op_new)

println("A size: ", size(A_new))
println("Cached after: ", op_new._matrix !== nothing)
u_new = A_new \ b_new
println("Max solution (≈ 0.073 for -Δu=1): ", round(maximum(u_new), digits=4))

MethodError: MethodError: no method matching assemble_weak_form(::TensorProductWeakForm, ::Tuple{Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.NodeToDofGlue{Int32}}, Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.CompressedCellConformity{FillArrays.Fill{Int64, 1, Tuple{Base.OneTo{Int64}}}}}}, ::Tuple{Gridap.CellData.GenericMeasure, Gridap.CellData.GenericMeasure})

Closest candidates are:
  assemble_weak_form(::TensorProductWeakForm, !Matched::Tuple{Vararg{var"#s26", N}} where var"#s26"<:FESpace, ::Tuple{Vararg{Measure, N}}) where N
   @ GridapTensorProduct ~/Documents/TU-Delft/WP2/SWE_farfield/TensorProduct/GridapTensorProduct.jl/src/TensorProductFEMOperators.jl:685


### Auto-detection: mass operator

A mass bilinear form evaluates to `M_k` on each subdomain (same as the reference mass), so
the product rule `M₁⊗M₂` is applied automatically.

In [14]:
a_mass(u, v, dΩ) = ∫(u * v) * dΩ
l_unit(v, dΩ)   = ∫(1.0 * v) * dΩ

op_mass = TensorProductAffineOperator(a_mass, l_unit, U_tp, V_tp)
M_auto  = get_matrix(op_mass)

# Compare with explicit assembly from low-level API
ops     = assemble_subdomain_operators((Vx, Vy), (dΩx, dΩy))
M_ref   = assemble_mass_tensor(ops)

err_mass = maximum(abs, M_auto - M_ref)
println("Mass auto vs. explicit max-diff: ", round(err_mass, sigdigits=3))  # ≈ 0

MethodError: MethodError: no method matching assemble_weak_form(::TensorProductWeakForm, ::Tuple{Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.NodeToDofGlue{Int32}}, Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.CompressedCellConformity{FillArrays.Fill{Int64, 1, Tuple{Base.OneTo{Int64}}}}}}, ::Tuple{Gridap.CellData.GenericMeasure, Gridap.CellData.GenericMeasure})

Closest candidates are:
  assemble_weak_form(::TensorProductWeakForm, !Matched::Tuple{Vararg{var"#s26", N}} where var"#s26"<:FESpace, ::Tuple{Vararg{Measure, N}}) where N
   @ GridapTensorProduct ~/Documents/TU-Delft/WP2/SWE_farfield/TensorProduct/GridapTensorProduct.jl/src/TensorProductFEMOperators.jl:685


---
## Composite forms — Helmholtz equation

For `−Δu + κ²u = f`, the bilinear form is the **sum** of two separable terms:
`a(u,v) = ∫(∇u⋅∇v)*dΩ + κ²∫(u*v)*dΩ`.

Pass a **list of 3-arg forms** — one per separable term. Each term is independently
evaluated on each subdomain and Kronecker-assembled. The results are summed.

In [15]:
κ² = 4.0

a_stiff = (u, v, dΩ) -> ∫(∇(u) ⋅ ∇(v)) * dΩ
a_mass2 = (u, v, dΩ) -> ∫(κ² * u * v) * dΩ

op_helm = TensorProductAffineOperator([a_stiff, a_mass2], l_unit, U_tp, V_tp)

A_helm  = get_matrix(op_helm)

# Verify: A_helm == stiffness_tensor + κ²·mass_tensor
A_ref_helm = assemble_stiffness_tensor(ops) .+ κ² .* assemble_mass_tensor(ops)
err_helm = maximum(abs, A_helm - A_ref_helm)
println("Helmholtz auto vs. explicit max-diff: ", round(err_helm, sigdigits=3))  # ≈ 0
println("A_helm symmetric: ", maximum(abs, A_helm - A_helm') < 1e-12)  # Helmholtz is symmetric

MethodError: MethodError: no method matching assemble_weak_form(::TensorProductWeakForm, ::Tuple{Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.NodeToDofGlue{Int32}}, Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.CompressedCellConformity{FillArrays.Fill{Int64, 1, Tuple{Base.OneTo{Int64}}}}}}, ::Tuple{Gridap.CellData.GenericMeasure, Gridap.CellData.GenericMeasure})

Closest candidates are:
  assemble_weak_form(::TensorProductWeakForm, !Matched::Tuple{Vararg{var"#s26", N}} where var"#s26"<:FESpace, ::Tuple{Vararg{Measure, N}}) where N
   @ GridapTensorProduct ~/Documents/TU-Delft/WP2/SWE_farfield/TensorProduct/GridapTensorProduct.jl/src/TensorProductFEMOperators.jl:685


### `TensorProductSeparableTerm` and `TensorProductWeakForm`

For more control, construct terms explicitly using `TensorProductSeparableTerm` and
combine them with `+`. This is equivalent to passing a list, but allows naming and
reusing individual terms.

In [16]:
using LinearAlgebra: norm

# Build named separable terms
term_stiff = TensorProductSeparableTerm((u, v, dΩ) -> ∫(∇(u)⋅∇(v)) * dΩ)
term_mass  = TensorProductSeparableTerm((u, v, dΩ) -> ∫(u * v) * dΩ)

# Compose with +  →  TensorProductWeakForm([term_stiff, term_mass])
wf = term_stiff + term_mass

println("TensorProductWeakForm with ", length(wf.terms), " terms")

# Assemble directly from the weak form
A_wf = assemble_weak_form(wf, (Vx, Vy), (dΩx, dΩy))

# Must match Helmholtz with κ²=1
A_ref_wf = assemble_stiffness_tensor(ops) .+ assemble_mass_tensor(ops)
println("WeakForm assembly max-diff: ", round(maximum(abs, A_wf - A_ref_wf), sigdigits=3))

TensorProductWeakForm with 2 terms


MethodError: MethodError: no method matching assemble_weak_form(::TensorProductWeakForm, ::Tuple{Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.NodeToDofGlue{Int32}}, Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.CompressedCellConformity{FillArrays.Fill{Int64, 1, Tuple{Base.OneTo{Int64}}}}}}, ::Tuple{Gridap.CellData.GenericMeasure, Gridap.CellData.GenericMeasure})

Closest candidates are:
  assemble_weak_form(::TensorProductWeakForm, !Matched::Tuple{Vararg{var"#s26", N}} where var"#s26"<:FESpace, ::Tuple{Vararg{Measure, N}}) where N
   @ GridapTensorProduct ~/Documents/TU-Delft/WP2/SWE_farfield/TensorProduct/GridapTensorProduct.jl/src/TensorProductFEMOperators.jl:685


### Advection–diffusion

Separable advection adds a `∫(v*(b·∇u))*dΩ` term. With scalar velocity components
`b=(bx, by)`, this is another sum-rule term (one advection matrix per subdomain direction).

In [17]:
bx, by = 2.0, 1.0

a_diff = (u, v, dΩ) -> ∫(∇(u) ⋅ ∇(v)) * dΩ
a_adv  = (u, v, dΩ) -> ∫(v * bx * (∇(u) ⋅ VectorValue(1.0))) * dΩ
# Note: each subdomain is 1D so ∇(u)⋅VectorValue(1.0) extracts the scalar derivative

op_adv_diff = TensorProductAffineOperator([a_diff, a_adv], l_unit, U_tp, V_tp)
A_adv_diff  = get_matrix(op_adv_diff)

println("Advection-diffusion A size:     ", size(A_adv_diff))
println("A symmetric (expect false):     ", maximum(abs, A_adv_diff - A_adv_diff') < 1e-12)

MethodError: MethodError: no method matching assemble_weak_form(::TensorProductWeakForm, ::Tuple{Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.NodeToDofGlue{Int32}}, Gridap.FESpaces.UnconstrainedFESpace{Vector{Float64}, Gridap.FESpaces.CompressedCellConformity{FillArrays.Fill{Int64, 1, Tuple{Base.OneTo{Int64}}}}}}, ::Tuple{Gridap.CellData.GenericMeasure, Gridap.CellData.GenericMeasure})

Closest candidates are:
  assemble_weak_form(::TensorProductWeakForm, !Matched::Tuple{Vararg{var"#s26", N}} where var"#s26"<:FESpace, ::Tuple{Vararg{Measure, N}}) where N
   @ GridapTensorProduct ~/Documents/TU-Delft/WP2/SWE_farfield/TensorProduct/GridapTensorProduct.jl/src/TensorProductFEMOperators.jl:685


---
## Legacy API — explicit `op_type` (backward compatible)

The old `op_type` keyword is fully supported. Pass a 2-argument bilinear form and
specify `op_type` explicitly. This triggers the original assembly path via
`assemble_subdomain_operators` (eager, all 5 matrices computed upfront).

In [18]:
# Legacy form: 2-argument bilinear form + explicit op_type
a_old(u, v) = ∫(∇(u) ⋅ ∇(v)) * dΩ_tp
l_old(v)    = ∫(1.0 * v)       * dΩ_tp

op_legacy = TensorProductAffineOperator(a_old, l_old, U_tp, V_tp;
                                         op_type    = :stiffness,
                                         quad_order = 2)
A_legacy = get_matrix(op_legacy)
b_legacy = get_vector(op_legacy)

# Results match the new auto-detection API
println("Legacy vs new API max-diff: ", round(maximum(abs, A_legacy - A_new), sigdigits=3))

UndefVarError: UndefVarError: `A_new` not defined

---
## Optional: custom per-subdomain RHS with `rhs_forms`

For separable non-unit forcing, supply per-subdomain 1D linear forms via `rhs_forms`.
The global RHS is assembled as `kron(b_N, ..., kron(b_2, b_1))`.
Compatible with both the new and legacy APIs.

In [19]:
# Separable forcing f(x,y) = x * y
f_x(x) = x[1]
f_y(y) = y[1]

op_custom = TensorProductAffineOperator(
    a_new, l_unit, U_tp, V_tp;
    quad_order = 2,
    rhs_forms  = (
        (v) -> ∫(f_x * v) * dΩx,
        (v) -> ∫(f_y * v) * dΩy
    )
)
b_custom = get_vector(op_custom)
println("Custom RHS assembled, length: ", length(b_custom))

TypeError: TypeError: in keyword argument rhs_forms, expected Union{Nothing, Tuple{Vararg{T, N}} where {N, T}}, got a value of type Tuple{var"#29#31", var"#30#32"}

---
## Extension to 3D: add a third subdomain

The API is **dimension-agnostic**. N=3 requires no code changes beyond adding a third space.

In [20]:
model_z = CartesianDiscreteModel((0.0, 1.0), 6)
Ωz = Interior(model_z)
dΩz = Measure(Ωz, quad_order)
Vz = TestFESpace(Ωz, reffe; conformity=:H1, dirichlet_tags="boundary")
Uz = TrialFESpace(Vz, 0.0)

V3 = TensorProductFESpace(Vx, Vy, Vz)
U3 = TensorProductFESpace(Ux, Uy, Uz)

# New API: auto-detection in 3D
a3_stiff(u, v, dΩ) = ∫(∇(u)⋅∇(v)) * dΩ
a3_mass(u, v, dΩ)  = ∫(u*v) * dΩ
l3(v, dΩ) = ∫(1.0*v) * dΩ

# Pure stiffness
op3 = TensorProductAffineOperator(a3_stiff, l3, U3, V3)
A3 = get_matrix(op3)
b3 = get_vector(op3)

println("3D system size:  ", size(A3))   # (7*7*5, 7*7*5) = (245, 245)
println("3D symmetric:    ", maximum(abs, A3 - A3') < 1e-13)

UndefVarError: UndefVarError: `reffe` not defined

---
## Connection map

```
CartesianDiscreteModel (×N 1D meshes)
       │
       ▼
 Measure(Ωk, q)  ──⊗──►  TensorProductMeasure            [Stage 1]
       │
       ▼
 TestFESpace / TrialFESpace  ──►  TensorProductFESpace     [Stage 2]
       │                          num_free_dofs = ∏ nf_k
       │
       ▼
 get_triangulation  ──►  TensorProductTriangulation        [Stage 3]
       │                   num_cells = ∏ nc_k
       │
       ▼
 assemble_subdomain_operators  ──►  SubdomainOperators{N}  [Stage 4a]
       │                             M, K, G, D, A per subdomain
       ▼
 assemble_stiffness_tensor (etc.)                          [Stage 4b]
       │   A = ∑_k kron(M_N,…,K_k,…,M_1)
       │
       ▼
 TensorProductAffineOperator → (A, b)                      [Stage 5]
       │
       ▼
       A \ b  →  u_dofs
```